In [1]:
!pip install pandas

In [2]:
from pathlib import Path

import pandas as pd

In [3]:
from dataclasses import dataclass

@dataclass
class TestResult:
    time_res: int
    test_name: str

def parse_file(file: Path) -> list[TestResult]:
    results: list[TestResult] = list()
    with open(file) as f:
        for line in f.readlines():
            tokens = list(line.split())
            if len(tokens) > 1 and tokens[1] in ["'single_search'", "tests", "'freqs_search'"]:
                time_data = float(tokens[-2])
                time_type = tokens[-1]
                if time_type == "milliseconds":
                    time_data *= 1000
                if time_type == "seconds":
                    time_data *= 1000 * 1000
                results.append(TestResult(int(time_data), tokens[1]))
    return results


In [4]:
root_dir = Path("bench_res")
df = pd.DataFrame(columns=['solution', 'test', 'subtest', 'time'])
for test in root_dir.iterdir():
    if test.is_dir():
        #if test.name not in ["main", "2_term_with_check_before_lead", "rec_search_tree", "2_term"]:
        #    continue
        for subtest in test.iterdir():

            results = parse_file(subtest)
            for val in results:
                row = pd.DataFrame([{
                    "solution": test.name,
                    "test": subtest.name,
                    "subtest": val.test_name,
                    "time": val.time_res
                }])
                df = pd.concat([df, row], ignore_index=True)

        

In [5]:
list_of_dicts = df.to_dict('records')

def find_record(list_of_dicts, record):
    for rec in list_of_dicts:
        find = True
        for key, val in record.items():
            if rec[key] != val:
                find = False
                break
        if find:
            return rec

for i in range(len(list_of_dicts)):
    val = list_of_dicts[i]
    main_var = find_record(list_of_dicts, {
        "solution": "main",
        "test": val["test"],
        "subtest": val["subtest"]
    })

    def time_change(time_before, time_after):
        return (time_after - time_before) / time_before * 100
    list_of_dicts[i]["time_change"] = f"{time_change(main_var["time"], val["time"]):.2f}%"
df = pd.DataFrame(list_of_dicts)
df

,solution,test,subtest,time,time_change
0,main,interval_bench_big_data_freqs_discrete.json__m...,'single_search',244,0.00%
1,main,interval_bench_big_data_freqs_discrete.json__m...,'freqs_search',1042,0.00%
2,main,interval_bench_big_data_freqs_discrete.json__m...,tests,1324,0.00%
3,main,interval_bench_big_data_freqs_discrete.json__f...,'single_search',506,0.00%
4,main,interval_bench_big_data_freqs_discrete.json__f...,'freqs_search',2043,0.00%
...,...,...,...,...,...
265,rec_search_tree,interval_bench_big_data_freqs_discrete.json__m...,'freqs_search',10175,787.87%
266,rec_search_tree,interval_bench_big_data_freqs_discrete.json__m...,tests,11448,688.97%
267,rec_search_tree,interval_bench_big_data_freqs_equal.json__fs__...,'single_search',5032,837.06%
268,rec_search_tree,interval_bench_big_data_freqs_equal.json__fs__...,'freqs_search',31236,1450.17%


In [7]:
df_filtered = df[df['test'] == 'interval_bench_big_data_freqs_discrete.json__mmap___1_5simd_results.txt']
print(df_filtered)

                          solution  \
0                             main   
1                             main   
2                             main   
54                          2_term   
55                          2_term   
56                          2_term   
108  2_term_with_check_before_lead   
109  2_term_with_check_before_lead   
110  2_term_with_check_before_lead   
162                         1_term   
163                         1_term   
164                         1_term   
216                rec_search_tree   
217                rec_search_tree   
218                rec_search_tree   

                                                  test          subtest  time  \
0    interval_bench_big_data_freqs_discrete.json__m...  'single_search'   244   
1    interval_bench_big_data_freqs_discrete.json__m...   'freqs_search'  1042   
2    interval_bench_big_data_freqs_discrete.json__m...            tests  1324   
54   interval_bench_big_data_freqs_discrete.json__m...  'single_sea

In [ ]:
df.to_csv('output.csv', index=False)
